# CellSight AI — Part E: Multi-omics fusion on the iHMP prediabetes cohort

Dataset: Zhou et al. 2019 (Nature) iHMP T2D cohort, processed matrices mirrored from the
MB-SupCon repository (github.com/ya61sen/MB-SupCon). 545 paired samples, 59 subjects,
724 blood metabolites + 96 gut-microbiome taxa. Task: predict insulin resistance (IR vs IS).

Verified results: naive split ~0.997 AUROC (inflated by leakage) -> subject-level split 0.69
-> subject-level + feature selection 0.78.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import roc_auc_score

BASE = "https://raw.githubusercontent.com/ya61sen/MB-SupCon/main/data/"
subjects = pd.read_csv(BASE + "subjects.csv", encoding="utf-8-sig")
met = pd.read_csv(BASE + "metabolome_abundance_csv.csv", on_bad_lines="skip")
gut = pd.read_csv(BASE + "gut_16s_abundance_csv.csv")
print(met.shape, gut.shape, subjects.shape)

In [ ]:
lab = subjects[subjects["IR_IS_classification"].isin(["IR", "IS"])][["SubjectID", "IR_IS_classification"]]

met["SubjectID"] = met["SampleID"].str.split("-").str[0]
gut["SubjectID"] = gut["SampleID"].str.split("-").str[0]

both = (met.drop(columns=["SubjectID"])
           .merge(gut.drop(columns=["SubjectID"]), on="SampleID"))
both["SubjectID"] = both["SampleID"].str.split("-").str[0]
both = both.merge(lab, on="SubjectID")
print("paired + labeled:", both.shape, "| subjects:", both["SubjectID"].nunique())
print(both["IR_IS_classification"].value_counts())

In [ ]:
y = (both["IR_IS_classification"] == "IR").astype(int).values
groups = both["SubjectID"].values
met_cols = [c for c in met.columns if c not in ["SampleID", "SubjectID"]]
gut_cols = [c for c in gut.columns if c not in ["SampleID", "SubjectID"]]
X_met = both[met_cols]
X_gut = both[gut_cols]
X_fus = pd.concat([X_met, X_gut], axis=1)

def pipe():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1))])

def run_cv(X, cv, groups=None):
    proba = cross_val_predict(pipe(), X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    return roc_auc_score(y, proba)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
gkf = GroupKFold(5)
print("NAIVE sample-level split (leaky):")
print("  metabolome:", round(run_cv(X_met, skf), 3),
      "| microbiome:", round(run_cv(X_gut, skf), 3),
      "| fused:", round(run_cv(X_fus, skf), 3))
print("SUBJECT-LEVEL split (honest):")
print("  metabolome:", round(run_cv(X_met, gkf, groups), 3),
      "| microbiome:", round(run_cv(X_gut, gkf, groups), 3),
      "| fused:", round(run_cv(X_fus, gkf, groups), 3))

The naive split is inflated because longitudinal visits from the same person appear in both
train and test folds. Subject-level (grouped) CV is the honest evaluation for longitudinal
omics data. Note: the published benchmark on this data used sample-level splits.

In [ ]:
def run_cv_fs(X, k):
    p = Pipeline([("impute", SimpleImputer(strategy="median")),
                  ("select", SelectKBest(mutual_info_classif, k=k)),
                  ("scale", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1))])
    proba = cross_val_predict(p, X, y, cv=GroupKFold(5), groups=groups, method="predict_proba")[:, 1]
    return roc_auc_score(y, proba)

print("SUBJECT-LEVEL, fused + per-fold feature selection:")
for k in [25, 50, 100, 200]:
    print(f"  top-{k}:", round(run_cv_fs(X_fus, k), 3))
print("reference, metabolome top-50:", round(run_cv_fs(X_met, 50), 3))

Fusion + feature selection gives the best honest score (~0.78). Takeaways: (1) grouped CV is
mandatory for longitudinal data; (2) fusion only helps once feature noise is controlled.
Next: host transcriptome layer (raw iHMP RNA-seq requires a heavier pipeline — good mentor topic).